In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ MSE = M = \frac{1}{N} \sum_{i=1}^N (y_i - p_i)^2 $$

$$ \frac{\partial M}{\partial p} = \frac{2}{N} (p - y) $$

$$ \frac{\partial M}{\partial y} = -\frac{2}{N} (p - y) $$

$$ \diamond \diamond \diamond $$

$$ \text{Let } p = xW+b $$


In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd


def mean_squared_error(p: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
  """Computes the mean squared error between predictions `p` and targets `y`."""
  
  return ((p - y) ** 2).mean()


class MeanSquaredErrorFunction(autograd.Function):
    """Custom autograd function for mean squared error."""

    @staticmethod
    def forward(ctx, p, y):
        ctx.save_for_backward(p, y)
        return mean_squared_error(p, y)
    
    @staticmethod
    def backward(ctx, dF_df):
        (p, y) = ctx.saved_tensors

        dM_dp = 2 * (p - y) / p.numel()
        dM_dy = -dM_dp
        
        return (dF_df * dM_dp, dF_df * dM_dy)
    

class MeanSquaredError(nn.Module):
    """Mean squared error loss module."""

    def __init__(self):
        super().__init__()
    
    def forward(self, p, y):
        return MeanSquaredErrorFunction.apply(p, y)
    

def test_mse_function_forward():
  p = tr.tensor([0.2, -1.0, 3.0, 2.5], dtype=tr.float32, requires_grad=True)
  y = tr.tensor([0.0, -2.0, 1.0, 2.0], dtype=tr.float32, requires_grad=True)

  actual = MeanSquaredErrorFunction.apply(p, y)
  expected = tr.nn.MSELoss()(p, y)

  tr.testing.assert_close(actual, expected)


def test_mse_function_backward():
  p = tr.tensor([0.2, -1.0, 3.0, 2.5], dtype=tr.float32, requires_grad=True)
  y = tr.tensor([0.0, -2.0, 1.0, 2.0], dtype=tr.float32, requires_grad=True)

  actual = MeanSquaredErrorFunction.apply(p, y)
  actual.backward()

  expected_dM_dp = 2 * (p.detach() - y.detach()) / p.shape[0]
  expected_dM_dy = -expected_dM_dp

  tr.testing.assert_close(p.grad, expected_dM_dp)
  tr.testing.assert_close(y.grad, expected_dM_dy)


def test_mse_function_gradcheck():
    p = tr.randn(4, dtype=tr.float64, requires_grad=True)
    y = tr.randn(4, dtype=tr.float64, requires_grad=True)

    def fn(p, y):
        return MeanSquaredErrorFunction.apply(p, y)

    tr.autograd.gradcheck(fn, (p, y), eps=1e-6, atol=1e-4, rtol=1e-4)


if __name__ == "__main__":
  test_mse_function_forward()
  test_mse_function_backward()
  test_mse_function_gradcheck()